In [ ]:
import pandas as pd
import scaffolding
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os
import asyncio
import zipfile
import tarfile
import glob

# from geopy.geocoders import Nominatim
# from tqdm import tqdm

import matplotlib.transforms as mtrans
import matplotlib.ticker as mticker
from matplotlib.ticker import FuncFormatter

import matplotlib.cm as cm
import matplotlib.colors as mcolors


from pprint import pprint

from plots import *


import emcommon.diary.base_modes as emcb
import emcommon.metrics.footprint.util as emcfu
import emcommon.metrics.footprint.footprint_calculations as emcfc
import emcommon.util as emcommonutil

In [24]:
# show all columns when viewing dataframes
pd.set_option('display.max_columns', None)


In [ ]:
lookupdictionary = {
    "sc": "sc_21",
    "boulder": "cc_21",
    "fortcollins": "fc_21",
    "pueblo": "pc_21",
    "durango": "4c_21",
    "vail": "vail_22",
}

# Initialize an empty DataFrame to store combined data
combined_confirmed_trips = pd.DataFrame()

# Loop through each program and read confirmed_trip.csv
#
# TODO: just do transit and mode confirm trips

#
for biggername in lookupdictionary:
    # Read the confirmed_trip.csv for the current program
    confirmed_trip = pd.read_csv(
        f"CanBikeCO_2/ceo_{lookupdictionary[biggername]}/analysis_confirmed_trip.csv"
    )
    
    # Add a column to identify the program (useful for tracking)
    confirmed_trip['program'] = biggername
    
    # Combine into a single DataFrame
    combined_confirmed_trips = pd.concat([combined_confirmed_trips, confirmed_trip], ignore_index=True)


combined_confirmed_trips.reset_index(drop=True, inplace=True)

print(len(combined_confirmed_trips))
print("Combined Confirmed Trips:")
combined_confirmed_trips.head(55)


In [ ]:
df_pur = pd.read_csv(r'auxiliary_files/purpose_labels.csv')
df_re = pd.read_csv(r'auxiliary_files/mode_labels.csv')
df_ei = pd.read_csv(r'auxiliary_files/energy_intensity.csv')

#dictionaries:
dic_pur = dict(zip(df_pur['purpose_confirm'],df_pur['bin_purpose'])) # bin purpose
dic_re  = dict(zip(df_re['replaced_mode'],df_re['mode_clean'])) # bin modes
dic_fuel = dict(zip(df_ei['mode'],df_ei['fuel']))

# convert a dictionary to a defaultdict
dic_re = defaultdict(lambda: 'Other',dic_re)
dic_pur = defaultdict(lambda: 'Other',dic_pur)
dic_fuel = defaultdict(lambda: 'Other',dic_fuel)


# map modes, replaced modes, and trip purposes
combined_confirmed_trips['Mode_confirm'] = combined_confirmed_trips['data_user_input_mode_confirm'].map(dic_re)
combined_confirmed_trips['Replaced_mode'] = combined_confirmed_trips['data_user_input_replaced_mode'].map(dic_re)
combined_confirmed_trips['Trip_purpose'] = combined_confirmed_trips['data_user_input_purpose_confirm'].map(dic_pur)

print(combined_confirmed_trips['data_user_input_mode_confirm'].unique())

# drop rows without a mode or marked as "Not a Trip"
combined_confirmed_trips = combined_confirmed_trips.dropna(subset=['data_user_input_mode_confirm'])
combined_confirmed_trips = combined_confirmed_trips[combined_confirmed_trips['Mode_confirm'] != 'Not a Trip']

# Filter rows where Mode_confirm is 'Bus' or 'Free Shuttle'
combined_confirmed_trips = combined_confirmed_trips[
    combined_confirmed_trips['Mode_confirm'].isin(['Bus', 'Free Shuttle'])
]


In [ ]:
print(len(combined_confirmed_trips))

In [ ]:
import pandas as pd
import emcommon.metrics.footprint.util as util


async def fetch_uace(row):
    """
    Fetch UACE code for a single row.
    """
    coords = [row['data_start_loc_longitude'], row['data_start_loc_latitude']]
    year = int(row['data_start_local_dt_year'])
    uace_result = await util.get_uace_by_coords(coords, year)
    return uace_result

async def add_uace_to_dataframe(df):
    """
    Fetch UACE code for each row and add it as a new column.
    """
    uace_list = []
    for _, row in df.iterrows():
        print(f"Fetching UACE for row {_}...")
        uace_code = await fetch_uace(row)
        print(f"Retrieved UACE: {uace_code}")
        uace_list.append(uace_code)


    df['UACE'] = uace_list
    return df

combined_confirmed_trips = await add_uace_to_dataframe(combined_confirmed_trips)


print("UACE column added:")
print(combined_confirmed_trips.head())


In [ ]:
# Get the most common UACEs and their percentage of appearance
uace_counts = combined_confirmed_trips['UACE'].value_counts(normalize=True) * 100

# Format the result as a DataFrame
uace_summary = uace_counts.reset_index()
uace_summary.columns = ['UACE', 'Percentage']
uace_summary['Percentage'] = uace_summary['Percentage'].round(2)  # Round to 2 decimal places

# Display the summary
print(uace_summary)


In [47]:
import logging

logging.getLogger("urllib3").setLevel(logging.ERROR)

In [ ]:
# modify default_json to include pilot_ebike
default_json = await emcfu.read_json_resource('label-options.default.json')
default_json['MODE'].append({"value": "pilot_ebike", "base_mode": "E_BIKE"})
labels = default_json

for index, row in combined_confirmed_trips.iterrows():

    trip_object = {
        "_id": row['_id'],
        "distance": row['data_distance'],
        "start_fmt_time": row['data_start_fmt_time'],
        "start_loc": {"coordinates": [row['data_start_loc_longitude'], row['data_start_loc_latitude']]},
        "user_input": {
            "mode_confirm": row['data_user_input_mode_confirm'],
            "replaced_mode_confirm": row['data_user_input_replaced_mode']
        }
    }
    footprint = await emcfc.calc_footprint_for_trip(trip_object, labels)
    replaced_footprint = await emcfc.calc_footprint_for_trip(trip_object, labels, 'replaced_mode') if not pd.isna(row['data_user_input_replaced_mode']) else {}
    
    if footprint[1]['ntd_uace_code'] != row['UACE']:
        print('!!!!!!!!', footprint[1]['ntd_uace_code'], row['UACE'])
        print(footprint)
        print('#'*50)
        print(trip_object)
#     print(footprint[1]['ntd_uace_code'])

In [ ]:
import pandas as pd

file_path = 'canbikeco_baclocs/ceo_pc_21/background_location.csv'

# Read only the column names
columns = pd.read_csv(file_path, sep='|', nrows=0).columns
print(list(columns))


In [ ]:
import pandas as pd

file_path = 'canbikeco_baclocs/ceo_pc_21/background_location.csv'

# Define the range of data_ts
lower_bound = "You have to put an int that is an epoch here"
upper_bound = "Ditto, you can find it in the dataframe"

# Chunk size for processing
chunk_size = 500000  

# Process CSV in chunks
filtered_results = []  # To collect filtered results

# Read all columns (do not use 'usecols')
for chunk in pd.read_csv(file_path, sep='|', chunksize=chunk_size):
    # Filter rows where data_ts is in the desired range
    filtered_chunk = chunk[(chunk['data_ts'] >= lower_bound) & (chunk['data_ts'] <= upper_bound) & (chunk['perno'] == 'This is a sensitive code here')]
    
    if not filtered_chunk.empty:
        filtered_results.append(filtered_chunk)

# Combine all filtered chunks into one DataFrame
if filtered_results:
    result_df = pd.concat(filtered_results)

    # Display all rows and columns in the resulting DataFrame
    pd.set_option('display.max_rows', None)  # Show all rows
    pd.set_option('display.max_columns', None)  # Show all columns
    pd.set_option('display.width', None)  # Expand column width for clarity

    # Print only the first 10 rows
    print("Filtered Rows (First 10):")
    print(result_df.head(10))

else:
    print("No matching rows found.")


In [9]:
import folium
def points_map(dataframe):

    #create a map
    this_map = folium.Map(prefer_canvas=True, control_scale = True)

    def plotDot(point):
        '''input: series that contains a numeric named latitude and a numeric named longitude
        this function creates a CircleMarker and adds it to your this_map'''
        folium.CircleMarker(location=[point.data_loc_latitude, point.data_loc_longitude],
                            radius=4,
                            weight=2).add_to(this_map)

    #use df.apply(,axis=1) to "iterate" through every row in your dataframe
    dataframe.apply(plotDot, axis = 1)

    #Set the zoom to the maximum possible
    this_map.fit_bounds(this_map.get_bounds())


    return this_map

In [ ]:
# this is going to break, but was left for future reference
# to use this in the future, whatsthis is a dataframe
# taken from analysis_recreated_location filtered ona particular data_section

# import folium

# mapit = None


# mapit.save( 'map.html')
map = points_map(whatsthis)
map.save('map.html')

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from datetime import datetime

# Suppose 'whatsthis' is your DataFrame with 'data_latitude', 'data_longitude', and 'data_ts'.
# Convert epoch to a readable format if desired, or just use the raw values.

# Extract the necessary data
lat = whatsthis['data_latitude'].values
lon = whatsthis['data_longitude'].values
ts = whatsthis['data_ts'].values

# Normalize timestamps to [0,1]
min_ts = ts.min()
max_ts = ts.max()
norm_ts = (ts - min_ts) / (max_ts - min_ts)

# Convert epoch to human-readable times if you want to annotate the colorbar
# For example, show actual times on the colorbar ticks:
start_time = datetime.utcfromtimestamp(min_ts)
end_time = datetime.utcfromtimestamp(max_ts)

# Create a figure and scatter plot
plt.figure(figsize=(10,6))
scatter = plt.scatter(lon, lat, c=norm_ts, cmap='RdYlBu', s=10)

# Add a colorbar
cbar = plt.colorbar(scatter)
cbar.set_label('Normalized Time')  # Label for the colorbar

# Optionally, customize the colorbar ticks to show real times
tick_positions = [0.0, 0.5, 1.0]
tick_times = [start_time, start_time + (end_time - start_time)*0.5, end_time]
tick_labels = [t.strftime('%Y-%m-%d %H:%M:%S') for t in tick_times]

cbar.set_ticks(tick_positions)
cbar.set_ticklabels(tick_labels)

plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Points Colored by Time')

# Save as a static image
plt.savefig('points_colored_by_time.png', dpi=300)
plt.show()

In [ ]:
points_map(result_df)